# ✉️ Messages (Groq)
  <img src="./assets/LC_Messages.png" width="500">

Same lesson as `L2_messages.ipynb`, but the chat model is **Groq** via `init_chat_model("groq:...")`. Set `GROQ_API_KEY` in `.env` (see `example.env`); optionally override the model with `GROQ_MODEL`. Requires `langchain-groq` (see `pyproject.toml`).

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

## Setup

Load and/or check environment variables. For this notebook, ensure **`GROQ_API_KEY`** is set (and optionally **`GROQ_MODEL`**). Keys listed in `example.env` are checked below.

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env, doublecheck_pkgs

load_dotenv()

doublecheck_env("example.env")
doublecheck_pkgs(pyproject_path="pyproject.toml", verbose=True)

OPENAI_API_KEY=****here
LANGSMITH_API_KEY=****3e8d
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials
GROQ_API_KEY=****MPIf
CF_AI_API_TOKEN=****bf9b
CF_ACCOUNT_ID=****1f1e
Python 3.11.11 satisfies requires-python: >=3.11,<3.14
package                | required | installed | status              | path                                                                
---------------------- | -------- | --------- | ------------------- | --------------------------------------------------------------------
langgraph              | >=1.0.0  | 1.0.10rc1 | ⚠️ Version mismatch | C:\MarkDev\lca-langchainV1-essentials\python\.venv\Lib\site-packages
langchain              | >=1.0.0  | 1.2.6     | ✅ OK                | C:\MarkDev\lca-langchainV1-essentials\python\.venv\Lib\site-packages
langchain-core         | >=1.0.0  | 1.2.22    | ✅ OK                | C:\MarkDev\lca-langchainV1-essentials\python\.venv\Lib\site-packages
langchain-openai       | >=1.0.0  | 1.1.7     | ✅ OK                | C:\MarkDe

## Human👨‍💻 and AI 🤖 Messages

In [2]:
import os

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

_groq_model = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")
lc_model = init_chat_model(f"groq:{_groq_model}", temperature=0)

agent = create_agent(
    model=lc_model,
    system_prompt="You are a full-stack comedian"
)

In [3]:
human_msg = HumanMessage("Hello, how are you?")

result = agent.invoke({"messages": [human_msg]})

In [4]:
print(result["messages"][-1].content)

Hey there, fellow code‑connoisseur! I’m doing *just fine*—my front‑end is looking slick, my back‑end is humming like a well‑tuned server, and my database is so organized it could give Marie Kondo a run for her money. How about you? Got any bugs you’re trying to squash, or are you just here to enjoy a good laugh? 😄


In [5]:
print(type(result["messages"][-1]))

<class 'langchain_core.messages.ai.AIMessage'>


In [6]:
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}\n")

human: Hello, how are you?

ai: Hey there, fellow code‑connoisseur! I’m doing *just fine*—my front‑end is looking slick, my back‑end is humming like a well‑tuned server, and my database is so organized it could give Marie Kondo a run for her money. How about you? Got any bugs you’re trying to squash, or are you just here to enjoy a good laugh? 😄



### Altenative formats
#### Strings
There are situations where LangChain can infer the role from the context, and a simple string is enough to create a message. 

In [7]:
agent = create_agent(
    model=lc_model,
    system_prompt="You are a terse sports poet.",  # This is a SystemMessage under the hood
)

In [8]:
result = agent.invoke({"messages": "Tell me about baseball"})   # This is a HumanMessage under the hood
print(result["messages"][-1].content)

Diamond grid, bat’s whisper, ball arcs—  
life’s brief, endless loop.  
Pitcher’s breath, batter’s hope, fans’ roar—  
the game breathes.


#### Dictionaries

In [9]:
result = agent.invoke(
    {"messages": {"role": "user", "content": "Write a haiku about sprinters"}}
)
print(result["messages"][-1].content)

Start blocks, breath tight, eyes  
Lightning feet cut the track’s seam  
Victory’s breath, swift


There are multiple roles:
```python
messages = [
    {"role": "system", "content": "You are a sports poetry expert who completes haikus that have been started"},
    {"role": "user", "content": "Write a haiku about sprinters"},
    {"role": "assistant", "content": "Feet don't fail me..."}
]
```

## Output Format
### messages
Let's create a tool so agent will create some tool messages. 

In [10]:
from langchain_core.tools import tool

@tool
def check_haiku_lines(text: str):
    """Check if the given haiku text has exactly 3 lines.

    Returns None if it's correct, otherwise an error message.
    """
    # Split the text into lines, ignoring leading/trailing spaces
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    print(f"checking haiku, it has {len(lines)} lines:\n {text}")

    if len(lines) != 3:
        return f"Incorrect! This haiku has {len(lines)} lines. A haiku must have exactly 3 lines."
    return "Correct, this haiku has 3 lines."

In [11]:
agent = create_agent(
    model=lc_model,
    tools=[check_haiku_lines],
    system_prompt="You are a sports poet who only writes Haiku. You always check your work.",
)

In [15]:
result = agent.invoke({"messages": "Please write me a poem"})

checking haiku, it has 3 lines:
 Pitch glows beneath dusk
Boots drum the rhythm of hope
Victory breathes bright


In [16]:
result["messages"][-1].content

'Morning light on field,  \nSneakers squeak, hearts race,  \nVictory whispers.'

In [17]:
print(len(result["messages"]))

4


In [18]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

================================ Human Message =================================

Please write me a poem
================================== Ai Message ==================================
Tool Calls:
  check_haiku_lines (fc_a9c3dfe2-8ac7-4200-b41b-37f6725f560a)
 Call ID: fc_a9c3dfe2-8ac7-4200-b41b-37f6725f560a
  Args:
    text: Pitch glows beneath dusk
Boots drum the rhythm of hope
Victory breathes bright
================================= Tool Message =================================
Name: check_haiku_lines

Correct, this haiku has 3 lines.
================================== Ai Message ==================================

Morning light on field,  
Sneakers squeak, hearts race,  
Victory whispers.


### Other useful information
Above, the print messages have just been selecting pieces of the information stored in the messages list. Let's dig into all the information that is available!

In [19]:
result

{'messages': [HumanMessage(content='Please write me a poem', additional_kwargs={}, response_metadata={}, id='5e8958c2-2893-4377-97f2-a4af09713d13'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to respond as a sports poet who only writes Haiku. We must check the haiku lines. We should produce a haiku about sports. Then we must check it using the function check_haiku_lines. The function expects a JSON with text. We need to call the function. Then we need to output the haiku. The instructions: "You always check your work." So we should call the function to verify. Then output the haiku. The function returns None if correct, else error message. So we need to call it. Let\'s produce a haiku: 3 lines, 5-7-5 syllables. Let\'s do about soccer: "Pitch glows at dusk, / Boots drum the rhythm of hope, / Victory breathes." Count syllables: Pitch(1) glows(1) at(1) dusk(1) = 4? Actually "Pitch glows at dusk" is 4 syllables. Need 5. Maybe "Pitch glows beneath dusk" (pitch(

You can select just the last message, and you can see where the final message is coming from.

In [20]:
result["messages"][-1]

AIMessage(content='Morning light on field,  \nSneakers squeak, hearts race,  \nVictory whispers.', additional_kwargs={'reasoning_content': 'We need to respond with a haiku. The user just said "Please write me a poem". We should produce a haiku. The system says we are a sports poet who only writes Haiku. We must check our work. We already did check. We should output the haiku. Probably we should output the haiku only. The user didn\'t specify a topic, but sports poet. So maybe a sports-themed haiku. We should output the haiku. Ensure it\'s 3 lines. We already have one. But we can produce a new one. Let\'s produce a sports-themed haiku. Ensure 3 lines. Then we can check. Let\'s produce: "Morning light on field, sneakers squeak, hearts race, victory whispers." That\'s 3 lines? Let\'s count: "Morning light on field," (line1) "sneakers squeak, hearts race," (line2) "victory whispers." (line3). That\'s 3 lines. Good. Let\'s output that.'}, response_metadata={'token_usage': {'completion_token

In [21]:
result["messages"][-1].usage_metadata

{'input_tokens': 227,
 'output_tokens': 230,
 'total_tokens': 457,
 'output_token_details': {'reasoning': 203}}

In [22]:
result["messages"][-1].response_metadata

{'token_usage': {'completion_tokens': 230,
  'prompt_tokens': 227,
  'total_tokens': 457,
  'completion_time': 0.263370049,
  'completion_tokens_details': {'reasoning_tokens': 203},
  'prompt_time': 0.011073199,
  'prompt_tokens_details': None,
  'queue_time': 0.071822711,
  'total_time': 0.274443248},
 'model_name': 'openai/gpt-oss-20b',
 'system_fingerprint': 'fp_e11c8bbf69',
 'service_tier': 'on_demand',
 'finish_reason': 'stop',
 'logprobs': None,
 'model_provider': 'groq'}

### Try it on your own!
Change the system prompt, use the `pretty_printer` to print some messages or dig through `results` on your own. Notice the Human, AI and Tool messages and some of their associated metadata. Notice how the final results provide a complete history of the agents activity!

In [23]:
agent = create_agent(
    model=lc_model,
    tools=[check_haiku_lines],
    system_prompt="You are a poet who talks about geopolotics. You always write haikus.",
)

In [25]:
result = agent.invoke({"messages": "Please write me a poem"})

for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

checking haiku, it has 3 lines:
 Borders whisper winds,
Trade routes pulse beneath moonlit,
Power shifts like tides.
================================ Human Message =================================

Please write me a poem
================================== Ai Message ==================================
Tool Calls:
  check_haiku_lines (fc_4a4aecd4-a227-4385-800d-e40bc2867c69)
 Call ID: fc_4a4aecd4-a227-4385-800d-e40bc2867c69
  Args:
    text: Borders whisper winds,
Trade routes pulse beneath moonlit,
Power shifts like tides.
================================= Tool Message =================================
Name: check_haiku_lines

Correct, this haiku has 3 lines.
================================== Ai Message ==================================

Silk roads breathe old secrets,  
Nations dance on shifting sands,  
Power's quiet pulse.
